# 2017 SOXX 原始持仓篮子的既有价格身份覆盖
仅核对 SEC 原始30股表与两个封存 Qlib 仪器清单；不读市场收益。

In [1]:
import hashlib,json,os,re
from pathlib import Path
import qlib
from qlib.data import D
root=Path.cwd().resolve();card=root/'research/stage1_soxx_2017_basket_price_coverage_preflight_20261023';plan=json.loads((card/'plan.json').read_text());source=plan['source']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
for key,path_key in [('original_report_result_sha256','original_report_result'),('pit_provider_manifest_sha256','pit_provider_manifest'),('pit_price_manifest_sha256','pit_price_manifest'),('current_provider_manifest_sha256','current_provider_manifest')]:assert sha(root/source[path_key])==source[key]
report=json.loads((root/source['original_report_result']).read_text());rows=[x for x in report['reports'] if x['report_date']==source['report_date']];assert len(rows)==1 and rows[0]['status']=='numeric_source_closure_passed' and len(rows[0]['equities'])==30 and rows[0]['accepted_at_utc']==source['accepted_at_utc']
source_card=root/'research/stage1_soxx_2017_2018_original_concentration_source_20261023';source_summary=json.loads((source_card/'summary.json').read_text());assert source_summary['source_NAS_archive_sha256']==source['source_NAS_archive_sha256'] and source_summary['source_NAS_scoped_restore_verified']
lookup=[]
for name,alias in plan['fixed_name_aliases_for_inventory_lookup_only']:
 matches=[x for x in rows[0]['equities'] if name.lower() in re.sub(r'\s+',' ',x['security_original']).lower()]
 assert len(matches)==1,(name,len(matches))
 lookup.append({'original_security':matches[0]['security_original'],'inventory_alias':alias,'reported_equity_weight':matches[0]['reported_equity_weight']})
assert len({x['inventory_alias'] for x in lookup})==9
inventories={}
for key in ('pit_provider_manifest','current_provider_manifest'):
 provider=(root/source[key]).parent
 qlib.init(provider_uri=str(provider),region='us',kernels=1)
 symbols=set(D.list_instruments(D.instruments(market='all'),start_time='2017-06-07',end_time='2017-06-07',as_list=True))
 inventories[key]={'manifest_sha256':source[key.replace('manifest','manifest_sha256')],'listed_instruments':len(symbols),'alias_present':{x['inventory_alias']:x['inventory_alias'] in symbols for x in lookup}}
for x in lookup:
 x['present_pit']=inventories['pit_provider_manifest']['alias_present'][x['inventory_alias']]
 x['present_current']=inventories['current_provider_manifest']['alias_present'][x['inventory_alias']]
missing_both=[x for x in lookup if not x['present_pit'] and not x['present_current']]
result={'status':'2017_first_original_basket_existing_Qlib_price_identity_gap','plan_sha256':sha(card/'plan.json'),'report_date':source['report_date'],'accepted_at_utc':source['accepted_at_utc'],'original_equity_rows':30,'prespecified_aliases_checked':len(lookup),'aliases_absent_from_both_inventories':len(missing_both),'absent_alias_disclosed_weight_sum':sum(x['reported_equity_weight'] for x in missing_both),'lookup_rows':lookup,'inventories':inventories,'source_NAS_archive_sha256':source_summary['source_NAS_archive_sha256'],'source_NAS_scoped_restore_verified':True,'provider_NAS_restoration_not_asserted':True,'no_security_mapping_certified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
assert result['aliases_absent_from_both_inventories']>0
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':'needs_original_security_price_identities_even_with_2017_start','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','missing_weight_lower_bound':result['absent_alias_disclosed_weight_sum'],'model_fits':0,'native_accounts':0,'market_outcome_reads':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(summary['decision'],result['aliases_absent_from_both_inventories'],result['absent_alias_disclosed_weight_sum'])

[94873:MainThread](2026-10-23 05:32:59,399) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/23 05:32:59 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[94873:MainThread](2026-10-23 05:32:59,769) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[94873:MainThread](2026-10-23 05:32:59,770) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1')}


[94873:MainThread](2026-10-23 05:32:59,777) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[94873:MainThread](2026-10-23 05:32:59,777) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[94873:MainThread](2026-10-23 05:32:59,778) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_constituents')}


needs_original_security_price_identities_even_with_2017_start 9 0.14274518693874025


## 中文结论
# 2017 SOXX 原始篮子：现有成分价格身份仍不完整

只读 2017-03-31 原始 SOXX 30 股披露（SEC 于 2017-06-06T00:54:56Z 接受）及两个封存 Qlib 成分价格清单。在固定的九个原证券名称与历史代码查找别名中，`IDCC/CAVM/CREE/CY/IDTI/MXIM/MSCC/SIMO/XLNX` 均未出现在两份清单的 2017-06-07 仪器列表。其原披露权益权重合计 **14.2745%**，没有重新归一化。

这个数字只是**现有两份 Qlib 价格源的最低缺口**；查找别名尚不是已认证的旧证券到价格序列映射，也不能证明外部供应商没有数据。原始 SEC 报告来源已有 NAS 限定恢复；本卡只读取清单，尚未确认两个本地价格 provider 的 NAS 恢复资格，因此继续保留本地对象。0 收益读取、0 模型拟合、0 账户。

决定：`needs_original_security_price_identities_even_with_2017_start`。从 2017 年开始不能直接绕开完整篮子价格缺口；下跌贡献集中度路线仍需逐只旧证券身份及历史价格，不得删除缺失股或把后来沿用的代码当原证券。SOXX v2 目标未达成。
